In [1]:
%matplotlib inline


# Numerical libraries
import numpy as np   

# Import Linear Regression machine learning library
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso

from sklearn.metrics import r2_score

# to handle data in form of rows and columns 
import pandas as pd    

# importing ploting libraries
import matplotlib.pyplot as plt   

#importing seaborn for statistical plots
import seaborn as sns

In [2]:
mpg_df = pd.read_csv("car-mpg.csv")  
mpg_df = mpg_df.drop('car_name', axis=1)
mpg_df['origin'] = mpg_df['origin'].replace({1: 'america', 2: 'europe', 3: 'asia'})
mpg_df = pd.get_dummies(mpg_df, columns=['origin'])
mpg_df = mpg_df.replace('?', np.nan)
mpg_df = mpg_df.apply(lambda x: x.fillna(x.median()),axis=0)

# separate independent and dependent variables

In [3]:
# Copy all the predictor variables into X dataframe. Since 'mpg' is dependent variable drop it
X = mpg_df.drop('mpg', axis=1)

# Copy the 'mpg' column alone into the y dataframe. This is the dependent variable
y = mpg_df[['mpg']]


In [4]:
from sklearn import preprocessing

# scale all the columns of the mpg_df. This will produce a numpy array
X_scaled = preprocessing.scale(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)  

In [5]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.30, random_state=1)

# fit a simple linear model

In [6]:
regression_model = LinearRegression()
regression_model.fit(X_train, y_train)

for idx, col_name in enumerate(X_train.columns):
    print("The coefficient for {} is {}".format(col_name, regression_model.coef_[0][idx]))

The coefficient for cyl is 2.505951804938497
The coefficient for disp is 2.5357082860560585
The coefficient for hp is -1.7889335736325245
The coefficient for wt is -5.551819873098728
The coefficient for acc is 0.11485734803440936
The coefficient for yr is 2.9318465482116083
The coefficient for car_type is 2.977869737601941
The coefficient for origin_america is -0.5832955290165974
The coefficient for origin_asia is 0.3474931380432237
The coefficient for origin_europe is 0.37741646808688273


In [7]:
intercept = regression_model.intercept_[0]

print("The intercept for our model is {}".format(intercept))

The intercept for our model is 23.665107741982705


# Create a regularized RIDGE model and note the coefficients

In [8]:
ridge = Ridge(alpha=.3)
ridge.fit(X_train,y_train)
print ("Ridge model:", (ridge.coef_))
    

Ridge model: [[ 2.47057467  2.44494419 -1.78573889 -5.47285499  0.10115618  2.92319984
   2.94492098 -0.57949986  0.34667456  0.37344909]]


# Create a regularized LASSO model and note the coefficients

In [9]:
lasso = Lasso(alpha=0.2)
lasso.fit(X_train,y_train)
print ("Lasso model:", (lasso.coef_))

# Observe, many of the coefficients have become 0 indicating drop of those dimensions from the model

Lasso model: [ 0.          0.         -0.3475456  -4.01181473  0.          2.64248634
  1.07111166 -0.54724128  0.          0.        ]


## Let us compare their scores

In [10]:
print(regression_model.score(X_train, y_train))
print(regression_model.score(X_test, y_test))


0.8343770256960538
0.8513421387780066


In [11]:
print(ridge.score(X_train, y_train))
print(ridge.score(X_test, y_test))

0.8343617931312616
0.8518882171608502


In [12]:
print(lasso.score(X_train, y_train))
print(lasso.score(X_test, y_test))

0.8114389394513553
0.8547810865027448


In [ ]:
# More or less similar results but with less complex models.  Complexity is a function of variables and coefficients
## Note - with Lasso, we get equally good result in test though not so in training.  Further, the number of dimensions is much less
# in LASSO model than ridge or un-regularized model

# Let us generate polynomial models reflecting the non-linear interaction between some dimensions

In [13]:
from sklearn.preprocessing import PolynomialFeatures

In [14]:
poly = PolynomialFeatures(degree = 2, interaction_only=True)

#poly = PolynomialFeatures(2)

In [15]:
X_poly = poly.fit_transform(X_scaled)
X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size=0.30, random_state=1)
X_train.shape

(278, 56)

# Fit a simple non regularized linear model on poly features-

In [22]:
regression_model.fit(X_train, y_train)
print(regression_model.coef_[0])


[ 4.44769778e-13  6.30577752e+11 -4.47492397e+00 -2.21298991e+00
 -2.95033140e+00 -1.56024075e+00  3.01757650e+00  8.39601331e+11
  2.34601198e+11 -1.96144208e+12  1.41290963e+12 -1.33521271e+00
 -1.00688171e+00 -2.11929321e-01  2.93179321e+00 -1.99110031e+00
  2.02026800e+12  3.33770787e+12  1.13626798e+12  1.08457059e+12
  3.62701416e-01  1.96899414e-01 -5.69725037e-01  3.44805908e+00
 -2.03672791e+00  4.85446630e+11  4.00089884e+11  3.81886780e+11
  1.30035400e-01 -5.89195251e-01 -1.87219238e+00 -4.75494385e-01
  1.89261031e+11  1.55983004e+11  1.48886162e+11 -2.02560425e-01
  5.10131836e-01 -3.70889282e+00 -8.60877120e+11 -7.09507916e+11
 -6.77227055e+11  5.51605225e-01  1.85357666e+00 -2.06531739e+11
 -1.70216980e+11 -1.62472527e+11  3.01086426e-01 -7.77650680e+10
 -6.40915294e+10 -6.11755229e+10  5.03656792e+11 -5.33551152e+11
 -5.09275890e+11 -7.43757102e+10  2.53062431e+12 -1.37698721e+12]


In [23]:
ridge = Ridge(alpha=.3)
ridge.fit(X_train,y_train)
print ("Ridge model:", (ridge.coef_))

Ridge model: [[ 0.          3.73512981 -2.93500874 -2.13974194 -3.56547812 -1.28898893
   3.01290805  2.04739082  0.0786974   0.21972225 -0.3302341  -1.46231096
  -1.17221896  0.00856067  2.48054694 -1.67596093  0.99537516 -2.29024279
   4.7699338  -2.08598898  0.34009408  0.35024058 -0.41761834  3.06970569
  -2.21649433  1.86339518 -2.62934278  0.38596397  0.12088534 -0.53440382
  -1.88265835 -0.7675926  -0.90146842  0.52416091  0.59678246 -0.26349448
   0.5827378  -3.02842915 -0.36548074  0.5956112  -0.15941014  0.49168856
   1.45652375 -0.43819158 -0.20964198  0.77665496  0.36489921 -0.4750838
   0.3551047   0.23188557 -1.42941282  2.06831543 -0.34986402 -0.32320394
   0.39054656  0.06283411]]


In [24]:
print(ridge.score(X_train, y_train))
print(ridge.score(X_test, y_test))


0.914322570200336
0.8613398053698498


In [25]:
lasso = Lasso(alpha=0.1)
lasso.fit(X_train,y_train)
print ("Lasso model:", (lasso.coef_))


Lasso model: [ 0.         -0.         -0.         -1.59613165 -5.22452383 -0.
  2.86907439  0.03030592 -0.10514919  0.          0.         -0.
 -0.          0.          0.28971732 -0.          0.         -0.
  0.11457443 -0.          0.          1.15720495  0.          0.
 -0.          0.          0.         -0.          0.04724906  0.
 -0.6925298  -0.          0.          0.         -0.         -0.
 -0.         -0.67082659  0.         -0.         -0.          0.16918498
 -0.         -0.61771612  0.          0.36046427  0.         -0.37086554
  0.          0.         -0.         -0.          0.18165859 -0.
 -0.         -0.        ]


In [26]:
print(lasso.score(X_train, y_train))
print(lasso.score(X_test, y_test))


0.8900519684208551
0.880222844847697
